# 문제 9. 유·무인 재난지원팀의 침수·지형 격자 지도 복원

**개정판 v2 · 제한시간 30분 · 난이도 최상 목표 · Python3 · Jupyter · CPU · CSV 제출**

## 상황

유·무인 재난지원팀은 드론 영상에서 침수된 건물·도로와 일반 수면을 구별하고, 구역별 지형 지도를 작성하려고 합니다. 각 항공영상의 **7×7 격자 49칸**에 대해 가장 넓은 면적을 차지하는 종류를 예측하세요.

공개 재난 영상과 실제 픽셀 주석을 가공한 가상 MUM-T 지원 과제입니다. 원본은 군 영상이 아닙니다. 이 과제는 주석된 지형을 구분하며 실제 통행 가능성이나 안전성을 판정하지 않습니다.

## Data

| 파일 | 내용 |
|---|---|
| data/train.csv | 이미지 765장, image_id / image_path / scene_id / split |
| data/train_labels.csv | 학습 이미지의 37,485개 격자 정답, image_id / cell_id / label |
| data/test.csv | 예측 이미지 191장, image_id / image_path / scene_id |
| data/images/ | 224×224 RGB 이미지 956장 |
| data/sample_submission.csv | 9,359행의 제출 형식과 정확한 순서 |
| models/image_encoder.onnx | 선택적으로 사용할 수 있는 고정 사전학습 이미지 특징 추출 모델 |
| feature_helper.py | 이미지별 1,280차원 특징 추출 함수. 격자 정답이나 분류기는 포함하지 않음 |
| training_examples.jpg | 학습 이미지와 격자 정답의 시각화 |

### 격자 좌표

한 칸은 32×32픽셀입니다. 이미지 왼쪽 위부터 가로 방향으로 0~6, 다음 줄 7~13 순서로 번호를 부여합니다.

- `row = cell_id // 7`, `col = cell_id % 7`
- 픽셀 범위: 세로 `[32*row : 32*(row+1)]`, 가로 `[32*col : 32*(col+1)]`
- cell_id는 0~48입니다. CSV의 한 행은 이미지 한 장이 아니라 **격자 한 칸**입니다.

### 예측할 종류

| label | 의미 |
|---|---|
| 0 | 기타 지표: 나무·풀·차량·배경 등 아래 종류에 속하지 않는 영역 |
| 1 | 침수로 주석된 건물 또는 도로 |
| 2 | 비침수로 주석된 건물 또는 도로 |
| 3 | 일반 수면 또는 수영장 |

원본 픽셀 주석의 종류를 위 네 가지로 합친 뒤, 제공 이미지 좌표계로 최근접 보간하여 정렬하고 각 칸에서 가장 많은 픽셀의 종류를 정답으로 정했습니다. 동률이면 작은 label을 선택합니다. 한 칸에 침수 픽셀이 조금 있다는 이유만으로 1이 되지는 않습니다. 주석의 경계와 축소 과정에는 오차가 있을 수 있습니다.

### 학습·검증 단위

`split`은 fit 573장, valid 192장입니다. 같은 원본 영상에서 잘라낸 이미지들은 동일한 scene_id를 가지며 fit·valid·test 중 한 곳에만 있습니다. **격자 행을 무작위로 섞어서 학습·검증을 나누지 마세요.** 제공된 split을 사용하고 ID는 모델 입력에서 제외하세요. 서로 다른 원본 영상 사이의 지리적 중첩까지 제거했다는 뜻은 아닙니다.

## 초기 코드

데이터 로딩과 이미지 확인까지만 제공합니다.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

BASE_DIR = Path(".")
train = pd.read_csv(BASE_DIR / "data/train.csv")
train_labels = pd.read_csv(BASE_DIR / "data/train_labels.csv")
test = pd.read_csv(BASE_DIR / "data/test.csv")
sample_submission = pd.read_csv(BASE_DIR / "data/sample_submission.csv")
display(train.head())
display(train_labels.head())
display(Image.open(BASE_DIR / train.iloc[0]["image_path"]))


## 과제

1. 이미지와 격자 정답을 연결하고, 클래스 분포와 공간적 패턴을 확인하세요.
2. 각 격자를 표현하는 특징과 분류 모델을 구성하세요. 이미지 전체 특징을 사용할 수도 있지만 같은 이미지 안의 격자를 구분할 정보가 필요합니다.
3. fit으로 학습하고 valid에서 아래 mIoU를 평가하세요. 공간적 문맥, 특징, 클래스 불균형 처리 등 필요한 개선을 선택하세요.
4. 최종 설정을 선택한 뒤 학습 전체로 재학습하여 테스트의 모든 격자를 예측하세요. 특징 변환의 학습도 검증 단계에서는 fit에만 적용하세요.

### 제공 모델 사용 방법 — 선택 사항

`from feature_helper import extract_features`로 불러옵니다. `extract_features([BASE_DIR / p for p in train.image_path])`는 같은 이미지 순서로 `(765, 1280)` 배열을 반환합니다. RGB 변환·크기 조정·정규화·CPU 실행이 포함됩니다. 반환값은 **이미지 전체의 특징**이며 `(이미지, 격자, 특징)` 배열이 아닙니다. 격자 특징 추출이나 모델 학습은 직접 작성하세요. 새 모델 다운로드나 신경망 전체 재학습은 필요하지 않습니다.

### 평가: 4개 클래스의 mIoU — 높을수록 좋음

모든 테스트 격자를 모아서 클래스 c마다 `IoU_c = TP_c / (TP_c + FP_c + FN_c)`를 계산하고, 클래스 0~3의 IoU를 동일 비중으로 평균합니다. 분모가 0인 클래스의 IoU는 0입니다. 이미지별 점수를 먼저 평균하는 방식이 아닙니다. 이 점수는 원본 픽셀 단위가 아닌 **격자 단위 mIoU**입니다.

검증 예: `jaccard_score(y_valid, predictions, labels=[0,1,2,3], average="macro", zero_division=0)`

빈도가 높은 지형만 맞혀서는 높은 점수를 얻기 어렵습니다. 연습 점수는 `100*mIoU`이며 공식 합격선은 없습니다.

## 결과 저장

Notebook과 같은 폴더에 **submission.csv**를 저장하세요.

| image_id | cell_id | label |
|---|---:|---:|
| test.csv의 첫 이미지 ID | 0 | 0 |
| test.csv의 첫 이미지 ID | 1 | 3 |

위 label은 형식 예시입니다.

- 컬럼 순서: `image_id,cell_id,label`.
- test.csv의 이미지 순서마다 cell_id 0~48 순서로 **9,359행**. sample_submission.csv의 두 식별자 열과 완전히 같아야 합니다.
- label은 정수 0~3입니다. 확률, 결측값, 중복 `(image_id,cell_id)`, 추가 컬럼은 허용하지 않습니다.
- `index=False`로 저장하고 코드·실행 결과가 담긴 Notebook도 저장하세요. 별도 보고서는 없습니다.


In [ ]:
# 여기에 코드를 작성하세요.


## CSV 저장 예시

In [ ]:
# predicted_labels: sample_submission 순서의 길이 9359 정수 배열
# submission = sample_submission.copy()
# submission["label"] = predicted_labels
# submission.to_csv(BASE_DIR / "submission.csv", index=False, encoding="utf-8-sig")


## 파일 형식 확인

In [ ]:
path = BASE_DIR / "submission.csv"
if path.exists():
    result = pd.read_csv(path)
    assert result.columns.tolist() == ["image_id", "cell_id", "label"]
    assert len(result) == len(sample_submission)
    assert result[["image_id", "cell_id"]].equals(sample_submission[["image_id", "cell_id"]])
    assert not result.duplicated(["image_id", "cell_id"]).any()
    assert pd.api.types.is_integer_dtype(result["label"]) and result["label"].isin([0,1,2,3]).all()
    print("제출 형식 확인 완료. 점수 채점 또는 최종 제출은 아닙니다.")
else:
    print("풀이 후 submission.csv를 생성하세요.")
